# ETH model pipeline

In [1]:
# Load Packages
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('eth_ml_dataset.csv')
data.head(1)

,date,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,...,eth_ret_1d_lag3,btc_ret_1d_lag3,sentiment_balance_lag3,momentum_gap_7_lag3,dow_sin,dow_cos,eth_btc_ret_corr_30,vix_x_eth_vol,buy_signal_pressure,altcoin_pressure
0,2018-01-02,1313.699951,9.77,884.44397,14982.099609,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.781831,0.62349,NaN,0.0,NaN,NaN


In [ ]:
df = data.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df["y_eth_next_return"] = df["ETH"].shift(-1) / df["ETH"] - 1

# Features: numeric columns only — no ETH, no date
feature_cols = [c for c in df.columns if c not in ("date", "ETH", "y_eth_next_return")]
X = df[feature_cols]
y = df["y_eth_next_return"]

mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.6, 0.2, 0.2
n = len(X)
n_test = int(n * TEST_FRAC)
n_val = int(n * VAL_FRAC)
n_train = n - n_val - n_test  

X_train = X.iloc[:n_train]
y_train = y.iloc[:n_train]
X_val = X.iloc[n_train : n_train + n_val]
y_val = y.iloc[n_train : n_train + n_val]
X_test = X.iloc[n_train + n_val :]
y_test = y.iloc[n_train + n_val :]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", len(y_train), "| Val:", len(y_val), "| Test:", len(y_test), "| Total:", n)
print("Feature count:", X_train.shape[1])

Train: 1248 | Val: 416 | Test: 416 | Total: 2080
Feature count: 169


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression

MODEL_BUILDERS = {
    "Lasso": lambda: Lasso(),
    "Ridge": lambda: Ridge(),
    "ElasticNet": lambda: ElasticNet(),
}

param_grid = {
    "Lasso": {
        "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000],
        "model__max_iter": [15000],
    },
    "Ridge": {"model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]},
    "ElasticNet": {
        "model__alpha": [0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.2, 0.5, 0.8],
        "model__max_iter": [15000],
    },
}

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=15)
best_model = {}

for name, builder in MODEL_BUILDERS.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", builder()),
    ])
    grid_search = GridSearchCV(
        pipeline,
        param_grid[name],
        cv=tscv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)
    best_model[name] = grid_search.best_estimator_
    print(f"Best parameters for {name}: {grid_search.best_params_}")

val_rmse = {}
for name, est in best_model.items():
    pred_val = est.predict(X_val)
    val_rmse[name] = float(np.sqrt(mean_squared_error(y_val, pred_val)))
    print(f"{name} RMSE on validation (next-day return): {val_rmse[name]:.6f}")

selected_name = min(val_rmse, key=val_rmse.get)
print(f"\nSelected model (lowest validation RMSE): {selected_name}")

final_estimator = clone(best_model[selected_name])
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
final_estimator.fit(X_trainval, y_trainval)

/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for impu

Best parameters for Lasso: {'model__alpha': 0.1, 'model__max_iter': 15000}


/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for impu

Best parameters for Ridge: {'model__alpha': 10000}


/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [83]. At least one non-missing value is needed for impu

Best parameters for ElasticNet: {'model__alpha': 0.1, 'model__l1_ratio': 0.5, 'model__max_iter': 15000}
Lasso RMSE on validation (next-day return): 0.036431
Ridge RMSE on validation (next-day return): 0.036484
ElasticNet RMSE on validation (next-day return): 0.036431

Selected model (lowest validation RMSE): Lasso


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [ ]:
from sklearn.metrics import mean_squared_error

y_test_pred = final_estimator.predict(X_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
print(f"Final TEST RMSE — {selected_name} (next-day return): {test_rmse:.6f}")

baselines = {
    "LinearRegression": LinearRegression(),
    "Lasso(alpha=1)": Lasso(alpha=1),
    "Ridge(alpha=1)": Ridge(alpha=1),
}
print("\nBaseline validation RMSE (for reference, default hyperparameters):")
for label, est in baselines.items():
    baseline_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", est),
    ])
    baseline_pipe.fit(X_train, y_train)
    pred_v = baseline_pipe.predict(X_val)
    rmse_v = float(np.sqrt(mean_squared_error(y_val, pred_v)))
    print(f"  {label}: {rmse_v:.6f}")

Final TEST RMSE — Lasso (next-day return): 0.044538

Baseline validation RMSE (for reference, default hyperparameters):
  LinearRegression: 0.063618
  Lasso(alpha=1): 0.036431
  Ridge(alpha=1): 0.067430


In [ ]:

y_trainval_arr = np.concatenate([y_train, y_val])
mu_trainval = float(np.mean(y_trainval_arr))

pred_zero = np.zeros_like(y_test, dtype=float)
pred_mean = np.full_like(y_test, fill_value=mu_trainval, dtype=float)

rmse_zero = float(np.sqrt(mean_squared_error(y_test, pred_zero)))
rmse_mean = float(np.sqrt(mean_squared_error(y_test, pred_mean)))

print("TEST RMSE — constant models (next-day return):")
print(f"  Always 0:                {rmse_zero:.6f}")
print(f"  Always mean(train+val): {rmse_mean:.6f}  (mean = {mu_trainval:.6f})")
print(f"  Your model ({selected_name}): {test_rmse:.6f}")
if test_rmse < rmse_zero and test_rmse < rmse_mean:
    print("  → Lower RMSE than both constant baselines on this test period.")
elif test_rmse < rmse_mean:
    print("  → Lower RMSE than mean baseline (not necessarily than 0).")
else:
    print("  → Does not beat the constant baselines on RMSE for this test period.")

TEST RMSE — constant models (next-day return):
  Always 0:                0.044515
  Always mean(train+val): 0.044538  (mean = 0.002226)
  Your model (Lasso): 0.044538
  → Does not beat the constant baselines on RMSE for this test period.


### Simple backtest (test period only)

Long ETH when the model predicts **positive** next-day return; otherwise **cash** (0%). Compare to **buy-and-hold** (always long).  
Does not include fees or slippage — add a cost per trade if you need realism.

In [8]:
# Backtest on held-out TEST only (same predictions as Final TEST RMSE)
y_pred_test = final_estimator.predict(X_test)
y_act = np.asarray(y_test, dtype=float)

# Long-only rule: in the market when predicted next-day return > 0
pos_model = (y_pred_test > 0).astype(float)
strat_ret = pos_model * y_act

# Buy-and-hold: always invested
bh_ret = y_act

def total_return(r):
    r = np.asarray(r, dtype=float)
    return float(np.prod(1.0 + r) - 1.0)

def max_drawdown(wealth):
    """wealth: cumulative product path starting at 1.0"""
    peak = np.maximum.accumulate(wealth)
    dd = (wealth - peak) / peak
    return float(dd.min())

w_strat = np.cumprod(1.0 + strat_ret)
w_bh = np.cumprod(1.0 + bh_ret)

print("TEST period — cumulative metrics (simple, no fees):")
print(f"  Model strategy (long if pred>0) total return: {100 * total_return(strat_ret):.2f}%")
print(f"  Buy-and-hold ETH total return:               {100 * total_return(bh_ret):.2f}%")
print(f"  Max drawdown (strategy): {100 * max_drawdown(w_strat):.2f}%")
print(f"  Max drawdown (buy-hold):   {100 * max_drawdown(w_bh):.2f}%")
print(f"  Time in market (strategy): {100 * pos_model.mean():.1f}%")

TEST period — cumulative metrics (simple, no fees):
  Model strategy (long if pred>0) total return: -12.75%
  Buy-and-hold ETH total return:               -12.75%
  Max drawdown (strategy): -63.24%
  Max drawdown (buy-hold):   -63.24%
  Time in market (strategy): 100.0%
